# Seq2Seq 모델 Q&A + Chatbot 만들기

1. qna 데이터셋을 찾아서 처리해서 준비한다.( 전처리 전반)
2. 인코더디코더seq2seq(인코더+디코더) 모델 만들기
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습
4. 챗봇을 만든다(모델 추론 +while문)

In [2]:
# %pip install datasets

### BCCard-Finance-Kor-QnA
- 제작기관: BC Card
- 도메인: 금융 (카드, 결제, 금융서비스)
- 데이터 용량: 31250
- 목적: 금융 지식 기반 챗봇 및 금융 도메인 LLM 학습
- 특징: 금융 서비스 중심 질문(카드 결제, 발급, 금융 서비스, 제도, 연체 이자 등)


https://huggingface.co/datasets/BCCard/BCCard-Finance-Kor-QnA?utm_source=chatgpt.com&library=datasets

In [3]:
from datasets import load_dataset

ds = load_dataset("BCCard/BCCard-Finance-Kor-QnA")

c:\Users\rbgh0\miniconda3\envs\pystudy_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\rbgh0\miniconda3\envs\pystudy_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rbgh0\.cache\huggingface\hub\datasets--BCCard--BCCard-Finance-Kor-QnA. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. I

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 31250
    })
})

In [5]:
ds["train"][0]

{'instruction': '비씨카드는 어떤 회사인가요?',
 'output': "비씨카드(BC카드)는 1982년에 설립된 대한민국의 대표적인 카드사로, 주로 결제 프로세싱 서비스를 제공하는 회사입니다. 설립 초기에는 '은행신용카드협회'라는 이름으로 시작했으며, BC는 'Bank Credit'의 약자입니다. 이 협회는 조흥은행, 상업은행, 서울은행, 제일은행, 한일은행 등 주요 은행들이 공동으로 설립하여, 카드 결제 시스템을 효율적으로 관리하고, 국내 카드 결제 인프라를 구축하는 역할을 했습니다.\n\n비씨카드는 이후 독립적인 신용카드 회사로 성장하며, 카드 발급, 결제 처리, 가맹점 관리 등의 다양한 금융 서비스를 제공하게 되었습니다. 현재 비씨카드는 KT가 최대 주주로, KT의 빅데이터 및 AI 기술을 활용해 디지털 금융 서비스를 강화하고 있습니다. 비씨카드는 결제 프로세싱 시장에서 오랜 경험과 노하우를 바탕으로, 국내외에서 다양한 금융 서비스를 제공하며 카드사 중에서도 중요한 위치를 차지하고 있습니다.\n\n비씨카드는 단순히 카드 발급과 결제에 그치지 않고, 디지털 결제 플랫폼 '페이북'을 운영하며, 중금리 대출, 마이데이터 사업 등으로 서비스 범위를 확장하고 있습니다. 이러한 디지털 혁신은 비씨카드가 빅테크 기업들과의 경쟁에서도 경쟁력을 유지하기 위한 핵심 전략입니다. \n\n이처럼 비씨카드는 전통적인 카드사로서의 역할을 유지하면서도, 디지털 금융의 선두주자로 자리매김하기 위해 지속적으로 변화를 추구하고 있는 기업입니다."}

In [6]:
# 한글 토큰화
from konlpy.tag import Okt

# 토큰화 함수
def tokenizer(text):
    okt = Okt()
    tokens = okt.morphs(text)
    return tokens
# 불용어 제거 함수
def remove_stopwords(tokens):
    stopwords = './ko_stopwords.txt'
    with open(stopwords, 'r', encoding='utf-8') as f:
        stopword_list = [line.strip() for line in f]
    return [token for token in tokens if token not in stopword_list]

# 특수문자 (., ? , ! 등) 제외하고 제거함수 / 일단 킵
# def remove_special_characters(tokens):
#     special_characters = '()[]{}<>/\\|@#$%^&*-_=+`'
#     return [token for token in tokens if token not in special_characters]

# 모든 데이터 토큰화 이후 불용어 제거
def preprocess_data(data):
    preprocessed_data = []
    for item in data:
        question = item['instruction']
        answer = item['output']
        # 토큰화
        question_tokens = tokenizer(question)
        answer_tokens = tokenizer(answer)
        # 불용어 제거
        question_tokens = remove_stopwords(question_tokens)
        answer_tokens = remove_stopwords(answer_tokens)
        preprocessed_data.append({'question': question_tokens, 'answer': answer_tokens})
    return preprocessed_data
preprocessed_ds = preprocess_data(ds['train'])
print(preprocessed_ds[0])

{'question': ['비씨카드', '회사', '인가요', '?'], 'answer': ['비씨카드', '(', 'BC', '카드', ')', '1982년', '설립', '된', '대한민국', '대표', '적', '인', '카드', ',', '주로', '결제', '프로세싱', '서비스', '제공', '하는', '회사', '입니다', '.', '설립', '초기', '에는', "'", '은행', '신용카드', '협회', "'", '라는', '이름', '시작', '했으며', ',', 'BC', "'", 'Bank', 'Credit', "'", '약자', '입니다', '.', '협회', '조', '은행', ',', '상업', '은행', ',', '서울', '은행', ',', '은행', ',', '한', '은행', '주요', '은행', '공동', '설립', '하여', ',', '카드', '결제', '시스템', '효율', '적', '관리', '하고', ',', '국내', '카드', '결제', '인프라', '구축', '하는', '역할', '했습니다', '.', '\n\n', '비씨카드', '이후', '독립', '적', '인', '신용카드', '회사', '성장하며', ',', '카드', '발급', ',', '결제', '처리', ',', '가맹', '점', '관리', '다양한', '금융', '서비스', '제공', '하게', '되었습니다', '.', '현재', '비씨카드', 'KT', '최대', '주주', ',', 'KT', '빅데이터', 'AI', '기술', '활용', '해', '디지털', '금융', '서비스', '강화하고', '있습니다', '.', '비씨카드', '결제', '프로세싱', '시장', '오랜', '경험', '노하우', '바탕', ',', '국내외', '다양한', '금융', '서비스', '제공', '하며', '카드', '중', '에서도', '중요한', '위치', '차지', '하고', '있습니다', '.', '\n\n', '비씨카드', '단순히', '카드', '

In [7]:
# vocab 만들기
from collections import Counter

special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']

counter = Counter()

for item in preprocessed_ds:
    counter.update(item['question'])
    counter.update(item['answer'])
    
vocab = special_tokens + list(counter.keys())
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for idx, word in word2idx.items()}

print(word2idx['비씨'])
print(word2idx['카드'])

838
10


In [8]:
# 문장 정수 시퀀스
def sentence_to_sequence(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

tokenized_data = []

for sample in preprocessed_ds:
    q_ids = sentence_to_sequence(sample['question'], word2idx)
    a_ids = [word2idx['<SOS>']] + sentence_to_sequence(sample['answer'], word2idx) + [word2idx['<EOS>']]
    
    tokenized_data.append({
        'question_ids': q_ids,
        'answer_ids': a_ids
    })
print(tokenized_data[0])

{'question_ids': [4, 5, 6, 7], 'answer_ids': [1, 4, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 10, 19, 20, 21, 22, 23, 24, 25, 5, 26, 27, 13, 28, 29, 30, 31, 32, 33, 30, 34, 35, 36, 37, 19, 9, 30, 38, 39, 30, 40, 26, 27, 33, 41, 31, 19, 42, 31, 19, 43, 31, 19, 31, 19, 44, 31, 45, 31, 46, 13, 47, 19, 10, 21, 48, 49, 17, 50, 51, 19, 52, 10, 21, 53, 54, 25, 55, 56, 27, 57, 4, 58, 59, 17, 18, 32, 5, 60, 19, 10, 61, 19, 21, 62, 19, 63, 64, 50, 65, 66, 23, 24, 67, 68, 27, 69, 4, 70, 71, 72, 19, 70, 73, 74, 75, 76, 77, 78, 66, 23, 79, 80, 27, 4, 21, 22, 81, 82, 83, 84, 85, 19, 86, 65, 66, 23, 24, 87, 10, 88, 89, 90, 91, 92, 51, 80, 27, 57, 4, 93, 10, 61, 21, 94, 95, 19, 78, 21, 96, 30, 97, 98, 30, 99, 87, 19, 88, 100, 101, 19, 102, 103, 104, 23, 105, 106, 51, 80, 27, 78, 107, 4, 108, 109, 110, 111, 112, 89, 113, 114, 115, 116, 44, 117, 118, 26, 27, 119, 4, 120, 17, 18, 10, 121, 55, 114, 122, 19, 78, 66, 123, 124, 125, 126, 127, 128, 17, 129, 130, 51, 131, 110, 26, 27, 2]}


In [10]:
# padding
max_q_len = max(len(sample['question_ids']) for sample in tokenized_data)
max_a_len = max(len(sample['answer_ids']) for sample in tokenized_data)

def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

pad_idx = word2idx['<PAD>']
for sample in tokenized_data:
    sample['question_ids'] = pad_sequence(sample['question_ids'], max_q_len, pad_idx)
    sample['answer_ids'] = pad_sequence(sample['answer_ids'], max_a_len, pad_idx)

In [ ]:
# seq2seq 학습용구조
# seq2seq_data = []

# for sample in tokenized_data:
#     encoder_input = sample['question_ids']
#     decoder_input = sample['answer_ids'][:-1]
#     decoder_target = sample['answer_ids'][1:]
    
#     seq2seq_data.append({
#         'encoder_input': encoder_input,
#         'decoder_input': decoder_input,
#         'decoder_target': decoder_target
#     })

# print(seq2seq_data[0])

In [16]:
!pip install torch

In [18]:
# dataset/ dataloader 만들기
import torch

from torch.utils.data import Dataset, DataLoader

class ChatDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        return {
            'encoder_input': torch.tensor(sample['encoder_input'], dtype=torch.long),
            'decoder_input': torch.tensor(sample['decoder_input'], dtype=torch.long),
            'decoder_target': torch.tensor(sample['decoder_target'], dtype=torch.long)
        }
chat_dataset = ChatDataset(tokenized_data)
dataloader = DataLoader(chat_dataset, batch_size=32, shuffle=True)
